In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.s4lake_bronze"
SILVER = "workspace.s4lake_silver"

def salvar(df, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SILVER}.{tabela}")

bsid_df = spark.table(f"{BRONZE}.bsid")
bsad_df = spark.table(f"{BRONZE}.bsad")

faturas_venda_df = spark.table(f"{SILVER}.faturas_venda")

def tratar_titulos(df, status_nome):
    return df.select(
        F.col("BUKRS").alias("empresa"),
        F.col("GJAHR").alias("ano_fiscal"),
        F.col("BELNR").alias("cod_documento"),
        F.col("BUZEI").alias("item_documento"),
        F.col("KUNNR").alias("cod_cliente"),
        F.col("VBELN").alias("cod_fatura"),
        F.to_date("BUDAT", "yyyyMMdd").alias("data_lancamento"),
        F.to_date("BLDAT", "yyyyMMdd").alias("data_documento"),
        F.col("BLART").alias("tipo_documento"),
        F.col("DMBTR").cast("decimal(15,2)").alias("valor"),
        F.col("WAERS").alias("moeda"),
        F.col("ZTERM").alias("cond_pagamento"),
        F.to_date("ZFBDT", "yyyyMMdd").alias("data_base"),
        F.col("ZBD1T").cast("integer").alias("prazo_dias"),
        F.to_date(
            F.when(F.col("AUGDT") != "00000000", F.col("AUGDT")), "yyyyMMdd"
        ).alias("data_pagamento"),
        F.col("AUGBL").alias("doc_pagamento"),
        F.lit(status_nome).alias("status")
    ).withColumn("data_vencimento", F.date_add("data_base", F.col("prazo_dias")))

titulos_abertos = tratar_titulos(bsid_df, "aberto")
titulos_pagos = tratar_titulos(bsad_df, "pago")

titulos_unificados = titulos_abertos.unionByName(titulos_pagos)

faturas_chaves = faturas_venda_df.select("cod_fatura").distinct().withColumn("fatura_existe", F.lit(True))

titulos_com_fatura = titulos_unificados.join(
    faturas_chaves,
    on="cod_fatura",
    how="left"
)

titulos_avaliados = titulos_com_fatura.withColumn(
    "motivo_rejeicao",
    F.when(F.col("fatura_existe").isNull(), F.lit("fatura de origem inexistente na VBRK"))
     .when((F.col("status") == "pago") & F.col("data_pagamento").isNull(), F.lit("título compensado sem data de pagamento"))
     .when((F.col("status") == "aberto") & F.col("data_pagamento").isNotNull(), F.lit("título em aberto com data de pagamento"))
     .otherwise(None)
).drop("fatura_existe")

titulos_validos = titulos_avaliados.filter(F.col("motivo_rejeicao").isNull()).drop("motivo_rejeicao")
titulos_quarentena = titulos_avaliados.filter(F.col("motivo_rejeicao").isNotNull()).withColumn("_quarentena_em", F.current_timestamp())

salvar(titulos_validos, "titulos_receber")
salvar(titulos_quarentena, "quarentena_titulos_receber")

In [0]:
spark.sql(f"""
    SELECT count(*) AS total_titulos_receber
    FROM {SILVER}.titulos_receber
""").show()

spark.sql(f"""
    SELECT status, count(*) AS total
    FROM {SILVER}.titulos_receber
    GROUP BY status
""").show()

spark.sql(f"""
    SELECT count(*) AS vencimentos_nulos
    FROM {SILVER}.titulos_receber
    WHERE data_vencimento IS NULL
""").show()

spark.sql(f"""
    SELECT count(*) AS itens_quarentena
    FROM {SILVER}.quarentena_titulos_receber
""").show()

spark.sql(f"""
    SELECT motivo_rejeicao, count(*) AS itens
    FROM {SILVER}.quarentena_titulos_receber
    GROUP BY motivo_rejeicao
""").show(truncate=False)

spark.sql(f"""
    SELECT 
        (SELECT count(*) FROM {SILVER}.titulos_receber) AS total_titulos,
        (SELECT count(*) FROM {SILVER}.faturas_venda) AS total_faturas
""").show()

In [0]:
spark.sql(f"""
    SELECT 
        silver_validos_quarentena,
        bronze_bsid_bsad,
        CASE WHEN silver_validos_quarentena = bronze_bsid_bsad
            THEN 'OK' ELSE 'DIFERENTE' 
        END AS validacao
    FROM (
        SELECT 
            (SELECT count(*) FROM {SILVER}.titulos_receber) + (SELECT count(*) FROM {SILVER}.quarentena_titulos_receber) AS silver_validos_quarentena,
            (SELECT count(*) FROM {BRONZE}.bsid) + 
            (SELECT count(*) FROM {BRONZE}.bsad) 
            AS bronze_bsid_bsad
    )
""").show()